In [5]:
import sys
sys.path.insert(0, '..')
from popularity_queries import pytrends_query, wiki_pageviews

import astropy
import astroquery
import pytrends
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from astroquery.simbad import Simbad
from urllib.parse import quote

In [4]:
#reading view data for last 5 years
planets = ["Mercury_(planet)", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]
views = wiki_pageviews(planets, start_date = "2021010100", end_date = "2026010100")

#needs to be a real date for data extraction
views.index = pd.to_datetime(views.index, format="%Y%m%d%H")
views.head()


,Mercury_(planet),Venus,Earth,Mars,Jupiter,Saturn,Uranus,Neptune
2021-01-01,3973,4677,8657,6370,6330,4903,3440,3565
2021-01-02,4562,5719,9772,7033,6680,5483,4238,4378
2021-01-03,4903,6010,10590,7243,6534,5461,4513,4567
2021-01-04,4326,5216,12696,11509,5815,4712,3810,3856
2021-01-05,4275,5208,11395,9750,6705,4545,3723,3808


In [11]:
ROLLING_BASELINE_DAYS = 28
MAD_MULTIPLIER = 6
SPIKE_RATIO = 10
MIN_SPIKE_DAYS = 3
RECOVERY_RATIO = 2
RECOVERY_WINDOW_DAYS = 14
RECOVERY_CONSECUTIVE_AVERAGES = 3

def consecutive_runs(mask):
    runs, start = [], None
    for i, v in enumerate(mask.to_numpy()):
        if v and start is None:
            start = i
        elif not v and start is not None:
            runs.append((start, i-1))
            start = None
    if start is not None:
        runs.append((start, len(mask)-1))
    return runs

In [ ]:
rows = []
for planet in planets:
    s = views[planet]

    #find normal baseline views
    baseline = max(float(s.median()),1.0)
    rolling = s.rolling(window=3, min_periods=3).mean()
    local_median = s.rolling(window=ROLLING_BASELINE_DAYS, min_periods=14).median().shift(3)
    local_mad = (s-local_median).abs().rolling(window=ROLLING_BASELINE_DAYS, min_periods=14).median()
    robust_sd = (1.4826 * local_mad).clip(lower=1)

    ratio = rolling >= SPIKE_RATIO * baseline
    robust_condition = rolling > local_median + MAD_MULTIPLIER * robust_sd
    potential_spike = (ratio & robust_condition).fillna(False)

    runs = [(a,b) for a, b in consecutive_runs(potential_spike) if b-a +1 >= MIN_SPIKE_DAYS]

    for a, b in runs:
        peak_avg = rolling.iloc[a:b + 1].max()
        # is it back to normal quickly or is it a new normal
        future = rolling.iloc[b+1:b+1 + RECOVERY_WINDOW_DAYS]
        normal = (future <= RECOVERY_RATIO * baseline).fillna(False)
        recovered = normal.rolling(RECOVERY_CONSECUTIVE_AVERAGES).sum().ge(RECOVERY_CONSECUTIVE_AVERAGES).any()

        window = s.iloc[max(a-2,0): b+1]
        peak_day = window.idxmax()

        rows.append({"planet": planet, "start": window.index[0].date(), "end": window.index[-1].date(),
                    "date": peak_day.date(), "views": s[peak_day], "normal": int(baseline),
                    "times_normal": round(peak_avg/baseline, 1), "recovered": bool(recovered),})

columns = ["planet", "start", "end", "date", "views", "normal", "times_normal", "recovered"]
spikes = pd.DataFrame(rows, columns=columns).sort_values("times_normal", ascending=False)
spikes

,planet,start,end,date,views,normal,times_normal,recovered
0,Mars,2021-02-24,2021-02-28,2021-02-26,201779,5402,17.8,False


In [13]:
#mars is the only planet that has a spike in our threshold
mars = views [["Mars"]].reset_index()
mars.columns = ["date", "views"]
mars.to_csv("Mars_daily_views.csv", index=False)

spikes[spikes["planet"] == "Mars"].to_csv("mars_spikes.csv", index=False)

In [20]:
#updating csv to have reasons, i will add to this as I find more and more celestial objects that have spikes within our threshold
reasons = {("Mars", "2021-02-26"): ("Mission", "NASA's Perseverance rover completes its first 10 days on Mars"),}

spikes["spike_type"] = ""
spikes["reason"] = ""

for i, row in spikes.iterrows():
    key = (row["planet"], str(row["date"]))
    if key in reasons:
        spikes.loc[i, "spike_type"], spikes.loc[i, "reason"] = reasons[key]

spikes.to_csv("reasons_for_planet_spikes.csv", index=False)
spikes

,planet,start,end,date,views,normal,times_normal,recovered,spike_type,reason
0,Mars,2021-02-24,2021-02-28,2021-02-26,201779,5402,17.8,False,Mission,NASA's Perseverance rover completes its first ...
